### Libraries

In [2]:
import amico
from os.path import join
import numpy as np
import subprocess

### Functions

In [3]:
# Function to convert neuroimaging files using mrconvert
def run_mrconvert(input_file, output_file, options=None):
    """
    Convert neuroimaging files using mrconvert.

    :param input_file: Path to the input file (e.g., .mif, .nii, .nii.gz).
    :param output_file: Path to the output file (desired format).
    :param options: List of additional command line options for mrconvert.
    """
    # Build the command as a list of strings
    command = ['mrconvert', input_file, output_file]

    # Add additional options if provided
    if options:
        command.extend(options)

    try:
        # Run the command and capture output and errors
        result = subprocess.run(command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        # Output the command's response for logging
        print(result.stdout.decode('utf-8'))

    except subprocess.CalledProcessError as e:
        print(f"Error: {e.stderr.decode('utf-8')}")

In [ ]:
sub_id = 'PNC019'

input_file = '/local_raid/data/pbautin/results/micapipe/micapipe_v0.2.0/sub-Pilot014/ses-01/dwi/sub-Pilot014_ses-01_space-dwi_desc-preproc_dwi.mif'  # Input neuroimaging file
output_dwi = '/local_raid/data/pbautin/results/COMMIT/sub-Pilot014_ses-01_space-dwi_desc-preproc_dwi.nii.gz'
output_bvec = '/local_raid/data/pbautin/results/COMMIT/dwi.bvec'  # Desired output file format
output_bval = '/local_raid/data/pbautin/results/COMMIT/dwi.bval'  # Desired output file format
options = ['-export_grad_fsl', output_bvec, output_bval]  # Optional arguments (e.g., setting strides)
run_mrconvert(input_file, output_dwi, options)
run_mrconvert(input_file, output_dwi)

In [6]:
sub_3Tid = 'sub-HC062'
sub_7Tid = 'sub-PNC19'
output_dwi = '/host/verges/tank/data/youngeun/myproject/FBA/3T_7T/subjects/PNC019_HC062/sub-HC062_ses-01_space-dwi_desc-preproc_dwi.nii.gz'
out_scheme = '/host/verges/tank/data/youngeun/myproject/FBA/3T_7T/subjects/PNC019_HC062/sub-HC062_ses-01_space-nativepro_dwi.scheme'

In [9]:
# Class to hold all the information (data and parameters) when performing an evaluation with the AMICO framework
# lmax : Maximum SH order to use for the rotation phase
# default output path: study_path/subject/AMICO/<MODEL>
ae = amico.Evaluation(study_path='/data/mica3/BIDS_MICs/derivatives/micapipe_v0.2.0', subject='sub-HC062', output_path='/host/verges/tank/data/youngeun/myproject/NODDI/3T/HC062')
amico.setup(lmax=12)

# Load the diffusion signal and its corresponding acquisition scheme.
#NODDI_img = join(ae.get_config("study_path"), ae.get_config("subject"), 'ses-01/dwi/sub-Pilot014_ses-01_space-dwi_desc-preproc_dwi.nii.gz')
# NODDI_scheme = join('/local_raid/data/pbautin/results/model_results/noddi', 'sub-Pilot014_ses-01_space-dwi_desc-preproc_dwi.scheme')
brain_mask = join(ae.get_config("study_path"), ae.get_config("subject"), 'ses-01/dwi/sub-HC062_ses-01_space-dwi_desc-brain_mask.nii.gz')
ae.load_data(output_dwi, out_scheme, mask_filename=brain_mask, b0_thr=10)

# Set the model to use to describe the signal contributions in each voxel.
# models: ['StickZeppelinBall', 'CylinderZeppelinBall', 'NODDI', 'FreeWater', 'SANDI']
ae.set_model('NODDI')

# Define NODDI model parameters to compute each compartment response function
# para_diff is the axial diffusivity (AD) in the CC -- single fiber
para_diff=1.7E-3
# iso_diff is the mean diffusivity (MD) in ventricles.
iso_diff=3.0E-3
intra_vol_frac = np.linspace(0.1, 0.99, 12)
intra_orient_distr = np.hstack((np.array([0.03, 0.06]), np.linspace(0.09, 0.99, 10)))
ae.model.set(dPar=para_diff, dIso=iso_diff,IC_VFs=intra_vol_frac, IC_ODs=intra_orient_distr, isExvivo=False)

# Generate the high-resolution response functions for each compartment with:
# lambda1 is the first regularization parameter.
# lambda2 is the second regularization parameter.
#        StickZeppelinBall:      'set_solver()' not implemented
#        CylinderZeppelinBall:   lambda1 = 0.0, lambda2 = 4.0
#        NODDI:                  lambda1 = 5e-1, lambda2 = 1e-3
#        FreeWater:              lambda1 = 0.0, lambda2 = 1e-3
#        VolumeFractions:        'set_solver()' not implemented
#        SANDI:                  lambda1 = 0.0, lambda2 = 5e-3
ae.set_solver(lambda1=5e-1, lambda2=1e-3)
ae.generate_kernels(regenerate=True)

# Load rotated kernels and project to the specific gradient scheme of this subject.
ae.load_kernels()
# Fit the model to the data.
ae.fit()
# Save the output (directions, maps etc).
ae.save_results()


-> Precomputing rotation matrices:
   [ DONE ]                                                                                                                                          

-> Loading data:
	* DWI signal
		- dim    = 140 x 140 x 92 x 156
		- pixdim = 1.600 x 1.600 x 1.600
	* Acquisition scheme
		- 156 samples, 10 shells
		- 4 @ b=0 , 24 @ b=2000.0 , 28 @ b=1990.0 , 33 @ b=1995.0 , 17 @ b=2005.0 , 7 @ b=300.0 , 1 @ b=305.0 , 2 @ b=295.0 , 8 @ b=705.0 , 23 @ b=700.0 , 9 @ b=695.0 
	* Binary mask
		- dim    = 140 x 140 x 92
		- pixdim = 1.600 x 1.600 x 1.600
		- voxels = 346737
   [ 5.3 seconds ]

-> Preprocessing:
	* Normalizing to b0... [ min=-22.94,  mean=0.73, max=110.40 ]
	* Keeping all b0 volume(s)
   [ 0.8 seconds ]

-> Creating LUT for "NODDI" model:
   [ 12.8 seconds ]                                                                                                                                  

-> Resampling LUT for subject "sub-HC062":
   [ 3.5 seconds ]        

In [10]:
import amico
import numpy as np
from os.path import join

# -------------------------------------------------------
# 1. Initialize AMICO Evaluation object
# -------------------------------------------------------
# study_path  : root directory where the DWI data is stored
# subject     : subject ID
# output_path : where AMICO results will be saved
ae = amico.Evaluation(
    study_path='/data/mica3/BIDS_MICs/derivatives/micapipe_v0.2.0',
    subject='sub-HC062',
    output_path='/host/verges/tank/data/youngeun/myproject/NODDI/3T/HC062'
)

# Setup spherical harmonics order
amico.setup(lmax=12)

# -------------------------------------------------------
# 2. Load DWI data, acquisition scheme, and brain mask
# -------------------------------------------------------
# This is where the brain mask is applied.
brain_mask = join(
    ae.get_config("study_path"),
    ae.get_config("subject"),
    'ses-01/dwi/sub-HC062_ses-01_space-dwi_desc-brain_mask.nii.gz'
)

# output_dwi and out_scheme must already be defined
ae.load_data(
    output_dwi,
    out_scheme,
    mask_filename=brain_mask,  # ★ Brain mask is applied here
    b0_thr=10
)

# -------------------------------------------------------
# 3. Set the model (NODDI)
# -------------------------------------------------------
ae.set_model('NODDI')

# -------------------------------------------------------
# 4. Configure NODDI model parameters
# -------------------------------------------------------
# dPar : axial diffusivity in the intracellular compartment
# dIso : isotropic diffusivity (e.g., CSF)
dPar = 1.7E-3
dIso = 3.0E-3

# Define the grid for intracellular volume fraction and orientation dispersion
IC_VFs = np.linspace(0.1, 0.99, 12)
IC_ODs = np.hstack((
    np.array([0.03, 0.06]),
    np.linspace(0.09, 0.99, 10)
))

ae.model.set(
    dPar=dPar,
    dIso=dIso,
    IC_VFs=IC_VFs,
    IC_ODs=IC_ODs,
    isExvivo=False
)

# -------------------------------------------------------
# 5. Set solver parameters and generate kernels
# -------------------------------------------------------
# lambda1 and lambda2 control regularization of the fitting
ae.set_solver(lambda1=5e-1, lambda2=1e-3)

# regenerate=True forces recomputation of response functions
ae.generate_kernels(regenerate=True)

# -------------------------------------------------------
# 6. Load and rotate kernels according to gradient scheme
# -------------------------------------------------------
ae.load_kernels()

# -------------------------------------------------------
# 7. Fit the model to the DWI data
# -------------------------------------------------------
ae.fit()

# -------------------------------------------------------
# 8. Save all results (maps, directions, etc.)
# -------------------------------------------------------
ae.save_results()



-> Precomputing rotation matrices:
   [ DONE ]                                                                                                                                          

-> Loading data:
	* DWI signal
		- dim    = 140 x 140 x 92 x 156
		- pixdim = 1.600 x 1.600 x 1.600
	* Acquisition scheme
		- 156 samples, 10 shells
		- 4 @ b=0 , 24 @ b=2000.0 , 28 @ b=1990.0 , 33 @ b=1995.0 , 17 @ b=2005.0 , 7 @ b=300.0 , 1 @ b=305.0 , 2 @ b=295.0 , 8 @ b=705.0 , 23 @ b=700.0 , 9 @ b=695.0 
	* Binary mask
		- dim    = 140 x 140 x 92
		- pixdim = 1.600 x 1.600 x 1.600
		- voxels = 346737
   [ 5.2 seconds ]

-> Preprocessing:
	* Normalizing to b0... [ min=-22.94,  mean=0.73, max=110.40 ]
	* Keeping all b0 volume(s)
   [ 0.8 seconds ]

-> Creating LUT for "NODDI" model:
   [ 11.1 seconds ]                                                                                                                                  

-> Resampling LUT for subject "sub-HC062":
   [ 3.2 seconds ]        

In [ ]:
fslswapdim /host/verges/tank/data/youngeun/myproject/NODDI/3T/HC062/AMICO/NODDI/FIT_ICVF.nii.gz \
           -x y z \
           /host/verges/tank/data/youngeun/myproject/NODDI/3T/HC062/FIT_ICVF_flippedLR.nii.gz